In [1]:
# ==========================================
# Flask Web Application - Digits Prediction
# ==========================================
# Place model.pkl and scaler.pkl in the same folder as this notebook/app.py.
# Run this cell, then open: http://127.0.0.1:5000

from flask import Flask, request, render_template_string
from werkzeug.utils import secure_filename
from PIL import Image, ImageOps, ImageDraw
import numpy as np
import joblib
import os
import base64
from io import BytesIO

app = Flask(__name__)

# -----------------------------
# Load trained model + scaler
# -----------------------------
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()

MODEL_PATH = os.path.join(BASE_DIR, "model.pkl")
SCALER_PATH = os.path.join(BASE_DIR, "scaler.pkl")

model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

MODEL_ACCURACY = 0.9833  # Accuracy obtained in the notebook evaluation

ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg", "bmp", "webp"}

def allowed_file(filename):
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS


def preprocess_image(file_storage):
    """
    Convert an uploaded digit image to the same 64-feature format
    expected by the sklearn Digits model.

    The sklearn Digits dataset uses 8x8 grayscale images with
    pixel values approximately in the range [0, 16].
    """
    image = Image.open(file_storage).convert("L")

    # Improve compatibility with common handwritten digit images:
    # if the background appears bright, invert to black background / white digit.
    arr = np.array(image)
    if arr.mean() > 127:
        image = ImageOps.invert(image)

    # Resize to the same spatial size as sklearn Digits: 8x8.
    image = image.resize((8, 8), Image.Resampling.LANCZOS)

    # Convert [0, 255] -> [0, 16].
    image_array = np.asarray(image, dtype=np.float64)
    image_array = (image_array / 255.0) * 16.0

    # Flatten to 64 features.
    features = image_array.reshape(1, 64)

    # Apply exactly the scaler used during training.
    features_scaled = scaler.transform(features)

    return image, features_scaled


def image_to_base64(image, add_prediction=None):
    """Return a browser-ready base64 PNG."""
    display_image = image.resize((320, 320), Image.Resampling.NEAREST).convert("RGB")

    if add_prediction is not None:
        draw = ImageDraw.Draw(display_image)
        draw.rectangle((0, 0, 320, 52), fill="black")
        draw.text(
            (12, 12),
            f"Prediction: {add_prediction}",
            fill="white"
        )

    buffer = BytesIO()
    display_image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Digits Prediction | MLflow Style</title>

    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            font-family: Inter, -apple-system, BlinkMacSystemFont, "Segoe UI",
                         Roboto, Arial, sans-serif;
            background: #f7f8fa;
            color: #202124;
        }

        .layout {
            display: flex;
            min-height: 100vh;
        }

        /* Sidebar */
        .sidebar {
            width: 250px;
            background: #17191c;
            color: #e8eaed;
            padding: 24px 18px;
            position: fixed;
            left: 0;
            top: 0;
            bottom: 0;
        }

        .brand {
            font-size: 21px;
            font-weight: 700;
            margin-bottom: 34px;
            padding: 0 10px;
        }

        .nav-title {
            color: #9aa0a6;
            font-size: 12px;
            text-transform: uppercase;
            letter-spacing: 0.08em;
            padding: 0 10px;
            margin-bottom: 10px;
        }

        .nav-item {
            display: block;
            padding: 11px 12px;
            border-radius: 7px;
            margin-bottom: 5px;
            color: #d7d9dc;
            background: #24272b;
        }

        .nav-item.active {
            background: #3a3f45;
            color: white;
        }

        .sidebar-info {
            position: absolute;
            left: 28px;
            right: 28px;
            bottom: 24px;
            color: #9aa0a6;
            font-size: 12px;
            line-height: 1.6;
        }

        /* Main */
        .main {
            margin-left: 250px;
            width: calc(100% - 250px);
            padding: 34px 44px 60px;
        }

        .header {
            max-width: 1100px;
            margin: 0 auto 28px;
        }

        .header h1 {
            margin: 0 0 7px;
            font-size: 28px;
        }

        .header p {
            margin: 0;
            color: #6b7077;
        }

        .content {
            max-width: 1100px;
            margin: 0 auto;
        }

        .card {
            background: white;
            border: 1px solid #e2e5e9;
            border-radius: 10px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.04);
            padding: 28px;
            margin-bottom: 22px;
        }

        .card-title {
            font-size: 17px;
            font-weight: 650;
            margin-bottom: 18px;
        }

        .upload-area {
            border: 2px dashed #c8cdd3;
            border-radius: 10px;
            padding: 48px 20px;
            text-align: center;
            transition: 0.2s;
        }

        .upload-area:hover {
            border-color: #6b7280;
            background: #fafafa;
        }

        .upload-icon {
            font-size: 40px;
            margin-bottom: 12px;
        }

        .upload-area h2 {
            margin: 0 0 8px;
            font-size: 19px;
        }

        .upload-area p {
            color: #70757d;
            margin: 0 0 20px;
        }

        input[type=file] {
            display: none;
        }

        .file-label {
            display: inline-block;
            background: #202124;
            color: white;
            padding: 11px 18px;
            border-radius: 6px;
            cursor: pointer;
            font-weight: 600;
        }

        .file-name {
            margin-top: 12px;
            color: #5f6368;
            font-size: 13px;
        }

        .predict-btn {
            margin-top: 18px;
            width: 100%;
            padding: 13px;
            border: 0;
            border-radius: 6px;
            background: #111315;
            color: white;
            font-size: 15px;
            font-weight: 650;
            cursor: pointer;
        }

        .predict-btn:hover {
            background: #30343a;
        }

        .result-grid {
            display: grid;
            grid-template-columns: 280px 1fr;
            gap: 28px;
            align-items: start;
        }

        .prediction-image {
            width: 100%;
            max-width: 280px;
            image-rendering: pixelated;
            border: 1px solid #dfe3e7;
            border-radius: 8px;
            background: #111;
        }

        .result-value {
            font-size: 54px;
            font-weight: 750;
            line-height: 1;
            margin: 6px 0 18px;
        }

        .metric {
            display: flex;
            justify-content: space-between;
            padding: 13px 0;
            border-bottom: 1px solid #eceef0;
        }

        .metric:last-child {
            border-bottom: none;
        }

        .metric-label {
            color: #70757d;
        }

        .metric-value {
            font-weight: 650;
        }

        .accuracy-bar {
            height: 9px;
            background: #e9ecef;
            border-radius: 20px;
            overflow: hidden;
            margin-top: 10px;
        }

        .accuracy-fill {
            height: 100%;
            width: {{ accuracy }}%;
            background: #343a40;
        }

        .error {
            padding: 14px 16px;
            border-radius: 7px;
            background: #fff1f1;
            color: #b42318;
            border: 1px solid #f3c7c7;
        }

        .hint {
            color: #7a7f86;
            font-size: 12px;
            margin-top: 12px;
            line-height: 1.5;
        }

        @media (max-width: 800px) {
            .sidebar {
                width: 200px;
            }

            .main {
                margin-left: 200px;
                width: calc(100% - 200px);
                padding: 25px;
            }

            .result-grid {
                grid-template-columns: 1fr;
            }
        }

        @media (max-width: 600px) {
            .sidebar {
                display: none;
            }

            .main {
                margin-left: 0;
                width: 100%;
                padding: 18px;
            }
        }
    </style>
</head>

<body>
<div class="layout">

    <aside class="sidebar">
        <div class="brand">Digits ML</div>

        <div class="nav-title">Workspace</div>
        <div class="nav-item active">Prediction</div>
        <div class="nav-item">Model Information</div>
        <div class="nav-item">Evaluation</div>

        <div class="sidebar-info">
            <strong>SVM · RBF</strong><br>
            sklearn Digits Dataset<br>
            10 classes · 8×8 input
        </div>
    </aside>

    <main class="main">

        <div class="header">
            <h1>Digits Prediction</h1>
            <p>Upload a handwritten digit image and run inference using the trained SVM model.</p>
        </div>

        <div class="content">

            <div class="card">
                <div class="card-title">Input Image</div>

                <form method="POST" enctype="multipart/form-data">
                    <div class="upload-area">
                        <div class="upload-icon">▧</div>
                        <h2>Upload a digit image</h2>
                        <p>PNG, JPG, JPEG, BMP or WEBP</p>

                        <label class="file-label" for="file">
                            Choose image
                        </label>

                        <input
                            id="file"
                            type="file"
                            name="file"
                            accept=".png,.jpg,.jpeg,.bmp,.webp"
                            required
                            onchange="document.getElementById('file-name').textContent = this.files[0]?.name || ''"
                        >

                        <div id="file-name" class="file-name"></div>
                    </div>

                    <button class="predict-btn" type="submit">
                        Predict Digit
                    </button>
                </form>

                <div class="hint">
                    The uploaded image is converted to grayscale, resized to 8×8,
                    scaled to the Digits dataset range, and then transformed with
                    the same StandardScaler used during training.
                </div>
            </div>

            {% if error %}
            <div class="card">
                <div class="error">{{ error }}</div>
            </div>
            {% endif %}

            {% if prediction is not none %}
            <div class="card">
                <div class="card-title">Prediction Result</div>

                <div class="result-grid">

                    <div>
                        <img
                            class="prediction-image"
                            src="data:image/png;base64,{{ image_data }}"
                            alt="Predicted digit"
                        >
                    </div>

                    <div>
                        <div class="metric-label">Predicted Class</div>
                        <div class="result-value">{{ prediction }}</div>

                        <div class="metric">
                            <span class="metric-label">Model</span>
                            <span class="metric-value">SVM (RBF)</span>
                        </div>

                        <div class="metric">
                            <span class="metric-label">Classes</span>
                            <span class="metric-value">0 – 9</span>
                        </div>

                        <div class="metric">
                            <span class="metric-label">Test Accuracy</span>
                            <span class="metric-value">{{ "%.2f"|format(accuracy) }}%</span>
                        </div>

                        <div class="accuracy-bar">
                            <div class="accuracy-fill"></div>
                        </div>
                    </div>

                </div>
            </div>
            {% endif %}

        </div>
    </main>
</div>
</body>
</html>
"""


@app.route("/", methods=["GET", "POST"])
def index():
    prediction = None
    image_data = None
    error = None

    if request.method == "POST":
        if "file" not in request.files:
            error = "No image was uploaded."
        else:
            file = request.files["file"]

            if file.filename == "":
                error = "Please choose an image."
            elif not allowed_file(file.filename):
                error = "Unsupported image format."
            else:
                try:
                    # Keep a sanitized filename for validation/logging.
                    filename = secure_filename(file.filename)

                    processed_image, features = preprocess_image(file)

                    prediction = int(model.predict(features)[0])

                    image_data = image_to_base64(
                        processed_image,
                        add_prediction=prediction
                    )

                except Exception as e:
                    error = f"Prediction failed: {str(e)}"

    return render_template_string(
        HTML,
        prediction=prediction,
        image_data=image_data,
        error=error,
        accuracy=MODEL_ACCURACY * 100
    )


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
